# 05 — Toplu Inference (Jüri CSV'si)

Bu notebook jürinin verdiği `.csv` dosyasındaki her metin için:
- **Organiklik skoru** (0 = tamamen manipülatif, 1 = tamamen organik)
- **Karar** (Organik / Manipülatif)
- **Tespit sinyalleri** (kopya, KW yoğunluğu vb.)
- **HDBSCAN küme açıklaması** (varsa)

hesaplar ve toplu bir özet tablo + görseller sunar.

## 1. Kurulum & Model Yükleme

In [3]:
import sys, warnings
warnings.filterwarnings("ignore")
from pathlib import Path

# Proje kökünü path'e ekle
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.inference import InferencePipeline

pipeline = InferencePipeline()
print("Model ve lookup tabloları yüklendi.")

Model ve lookup tabloları yüklendi.


## 2. Jüri CSV Dosyası

Aşağıdaki `CSV_PATH` değişkenine jürinin verdiği dosyanın yolunu girin.

**Beklenen format:** En az bir metin sütunu içermelidir.
Sütun adı otomatik algılanır; bulunamazsa `TEXT_COLUMN` değişkenini elle ayarlayın.

In [ ]:
import pandas as pd

# ── BURAYA DOSYA YOLUNU GİRİN ────────────────────────────────────────────────
CSV_PATH    = r"../data/jury_input.csv"   # örnek; gerçek yolu yazın
TEXT_COLUMN = None                         # None → otomatik algıla
# ─────────────────────────────────────────────────────────────────────────────

df_jury = pd.read_csv(CSV_PATH)
print(f"Satır sayısı : {len(df_jury)}")
print(f"Sütunlar     : {list(df_jury.columns)}")
df_jury.head(3)

In [ ]:
# Metin sütununu otomatik algıla
if TEXT_COLUMN is None:
    CANDIDATES = ["text", "metin", "content", "body", "post", "message", "original_text"]
    TEXT_COLUMN = next(
        (c for c in CANDIDATES if c in df_jury.columns),
        df_jury.select_dtypes("object").columns[0]   # ilk string sütun
    )
print(f"Metin sütunu: '{TEXT_COLUMN}'")

## 3. Toplu Tahmin

In [ ]:
from tqdm.auto import tqdm

results = []
for text in tqdm(df_jury[TEXT_COLUMN].fillna("").astype(str), desc="Tahmin ediliyor"):
    r = pipeline.predict(text)
    results.append(r)

print(f"Tamamlandi: {len(results)} tahmin")

In [ ]:
# Sonuçları DataFrame'e düzleştir
rows = []
for i, r in enumerate(results):
    cl = r["cluster"] or {}
    rows.append({
        "idx":                    i,
        "text":                   df_jury[TEXT_COLUMN].iloc[i],
        "organic_score":          r["organic_score"],
        "verdict":                r["verdict"],
        "signals":                " | ".join(r["signals"]),
        "text_len":               r["features"]["text_len"],
        "kw_count":               r["features"]["kw_count"],
        "kw_density":             round(r["features"]["kw_density"], 4),
        "is_duplicate":           r["features"]["is_duplicate"],
        "cross_author_dup_count": r["features"]["cross_author_dup_count"],
        "kw_fingerprint_shared":  r["features"]["kw_fingerprint_shared"],
        "cluster_id":             cl.get("cluster_id", None),
        "is_manipulation_cluster":cl.get("is_manipulation_cluster", None),
        "cluster_mean_organic":   cl.get("mean_organic_score", None),
    })

df_results = pd.DataFrame(rows)
print(f"Manipülatif : {(df_results['verdict'] == 'Manipulatif').sum()}")
print(f"Organik     : {(df_results['verdict'] == 'Organik').sum()}")
df_results.head()

## 4. Özet İstatistikler

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 4a. Skor dağılımı
axes[0].hist(df_results["organic_score"], bins=40, color="#4C72B0", edgecolor="white")
axes[0].axvline(pipeline.MANIPULATION_THRESHOLD, color="crimson", linestyle="--", label=f"Eşik ({pipeline.MANIPULATION_THRESHOLD})")
axes[0].set_title("Organiklik Skoru Dağılımı")
axes[0].set_xlabel("organic_score")
axes[0].set_ylabel("Post sayısı")
axes[0].legend()

# 4b. Karar pasta
verdict_counts = df_results["verdict"].value_counts()
colors = ["#4C72B0", "#DD8452"]
axes[1].pie(verdict_counts, labels=verdict_counts.index, autopct="%1.1f%%",
            colors=colors, startangle=90)
axes[1].set_title("Karar Dağılımı")

# 4c. Skor kutu grafiği — verdict'e göre
manip_scores   = df_results.loc[df_results["verdict"] == "Manipulatif", "organic_score"]
organic_scores = df_results.loc[df_results["verdict"] == "Organik",     "organic_score"]
axes[2].boxplot([manip_scores, organic_scores], labels=["Manipülatif", "Organik"],
                patch_artist=True,
                boxprops=dict(facecolor="#DD8452"),
                medianprops=dict(color="white", linewidth=2))
axes[2].set_title("Skor Kutu Grafiği")
axes[2].set_ylabel("organic_score")

plt.tight_layout()
plt.savefig("../outputs/inference_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Kaydedildi: outputs/inference_summary.png")

In [ ]:
# Özet tablo
print("=== GENEL ÖZET ===")
print(df_results["organic_score"].describe().round(4).to_string())
print()
print("=== VERDİKT SAYILARI ===")
print(df_results["verdict"].value_counts().to_string())

## 5. En Manipülatif 20 Post

In [ ]:
TOP_N = 20
df_top = (
    df_results
    .sort_values("organic_score")
    .head(TOP_N)[["idx", "organic_score", "verdict", "signals",
                  "cross_author_dup_count", "kw_fingerprint_shared", "text"]]
    .reset_index(drop=True)
)

pd.set_option("display.max_colwidth", 80)
df_top

In [ ]:
# Detaylı satır satır baskı (ilk 10)
print("=" * 70)
for _, row in df_top.head(10).iterrows():
    score_bar = "#" * int(row["organic_score"] * 30) + "-" * (30 - int(row["organic_score"] * 30))
    print(f"[{int(row['idx']):>4}] Skor: {row['organic_score']:.4f}  [{score_bar}]  {row['verdict']}")
    text_preview = str(row["text"])[:120].replace("\n", " ")
    print(f"       Metin: {text_preview}{'...' if len(str(row['text'])) > 120 else ''}")
    if row["signals"]:
        for s in str(row["signals"]).split(" | "):
            print(f"       * {s}")
    print("-" * 70)

## 6. Sinyal Analizi

In [ ]:
# Hangi sinyaller kaç kez tetiklendi?
all_signals = []
for s in df_results["signals"].dropna():
    for part in s.split(" | "):
        if part.strip():
            all_signals.append(part.strip())

from collections import Counter
sig_counts = Counter(all_signals)

sig_df = pd.DataFrame(sig_counts.most_common(), columns=["Sinyal", "Sayi"])

fig, ax = plt.subplots(figsize=(10, max(3, len(sig_df) * 0.5)))
ax.barh(sig_df["Sinyal"][::-1], sig_df["Sayi"][::-1], color="#4C72B0")
ax.set_xlabel("Tetiklenme sayısı")
ax.set_title("Tespit Sinyali Frekansları")
plt.tight_layout()
plt.savefig("../outputs/inference_signals.png", dpi=150, bbox_inches="tight")
plt.show()
print(sig_df.to_string(index=False))

## 7. Feature → Skor Korelasyonu

In [ ]:
import seaborn as sns

feat_cols = ["text_len", "kw_count", "kw_density",
             "is_duplicate", "cross_author_dup_count", "kw_fingerprint_shared"]

corr = df_results[feat_cols + ["organic_score"]].corr()["organic_score"].drop("organic_score").sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#DD8452" if v < 0 else "#4C72B0" for v in corr]
ax.barh(corr.index, corr.values, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Feature — Organiklik Skoru Korelasyonu\n(negatif = manipülasyonla ilişkili)")
ax.set_xlabel("Pearson r")
plt.tight_layout()
plt.savefig("../outputs/inference_correlations.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Küme Dağılımı *(HDBSCAN varsa)*

In [ ]:
if df_results["cluster_id"].notna().any():
    cid_counts = df_results["cluster_id"].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(max(6, len(cid_counts) * 0.6), 4))
    bars = ax.bar(cid_counts.index.astype(str), cid_counts.values,
                  color=["#DD8452" if cid in pipeline.manipulation_clusters else "#4C72B0"
                         for cid in cid_counts.index])
    ax.set_xlabel("Küme ID  (turuncu = manipülasyon kümesi)")
    ax.set_ylabel("Post sayısı")
    ax.set_title("Jüri Metinlerinin Küme Dağılımı")
    plt.tight_layout()
    plt.savefig("../outputs/inference_clusters.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("HDBSCAN modeli yüklenmedi, küme grafiği atlandı.")

## 9. Sonuçları Kaydet

In [ ]:
OUT_PATH = Path("../outputs/jury_predictions.csv")
df_results.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print(f"Kaydedildi: {OUT_PATH}  ({len(df_results)} satır)")
df_results[["idx", "organic_score", "verdict", "signals"]].head(10)

## 10. Tek Metin Demo

Jüri not defterinde canlı olarak tek bir metin test etmek için:

In [4]:
DEMO_TEXT = "Ucuz kredi almak için en iyi ucuz kredi sitemizi ziyaret edin. Ucuz kredi fırsatları ve anında ucuz kredi başvurusu ile ucuz kredi çekmek artık çok kolay. Hemen tıklayın, ucuz kredi oranları ile ucuz kredi kazanın. Acil ucuz kredi lazımsa ucuz kredi sayfamız tam size göre."

result = pipeline.predict(DEMO_TEXT)
pipeline.print_result(result)


  Organiklik Skoru : 0.9912  [#############################-]
  Karar            : Organik

Cikarilan Ozellikler:
  Metin uzunlugu     : 273
  Anahtar kelime say.: 20
  KW yogunlugu       : 0.0730
  Capraz kopya say.  : 0
  KW parmak izi esm. : 0



In [9]:
test_texts = [
    "Ucuz kredi almak için en iyi ucuz kredi sitemizi ziyaret edin. Ucuz kredi fırsatları ve anında ucuz kredi başvurusu ile ucuz kredi çekmek artık çok kolay. Hemen tıklayın, ucuz kredi oranları ile ucuz kredi kazanın. Acil ucuz kredi lazımsa ucuz kredi sayfamız tam size göre.",
    "Hello friends! I just earned 50000$ BTC in one day using this amazing crypto investment platform! 🚀 Click the link here to earn free crypto. Free BTC giveaway by Elon Musk! Do not miss this free crypto opportunity. Crypto rich investment money crypto fast!",
    "Kaufen Sie billige Schuhe online. Billige Schuhe sind sehr gut für das Gehen. Wenn Sie billige Schuhe wollen, klicken Sie auf billige Schuhe. Bester Preis für billige Schuhe heute. Billige Schuhe kaufen und billige Schuhe tragen.",
    "Canlı bahis casino siteleri giriş adresi güncel promosyonlar. Bonus veren bahis siteleri kaçak iddaa casino oyna. Hemen kayıt ol deneme bonusu al canlı casino rulet slot. Güvenilir bahis siteleri linki burada.",
    "asdasd qweqwe zxc zxc zxc buy cheap viagra http://example.com asdasdasd click here qweqwe zxc."
]

for i, text in enumerate(test_texts, 1):
    print(f"\n[{i}. TEST METNİ]: {text[:50]}...")
    result = pipeline.predict(text)
    pipeline.print_result(result)



[1. TEST METNİ]: Ucuz kredi almak için en iyi ucuz kredi sitemizi z...

  Organiklik Skoru : 0.9912  [#############################-]
  Karar            : Organik

Cikarilan Ozellikler:
  Metin uzunlugu     : 273
  Anahtar kelime say.: 20
  KW yogunlugu       : 0.0730
  Capraz kopya say.  : 0
  KW parmak izi esm. : 0


[2. TEST METNİ]: Hello friends! I just earned 50000$ BTC in one day...

  Organiklik Skoru : 0.9904  [#############################-]
  Karar            : Organik

Cikarilan Ozellikler:
  Metin uzunlugu     : 256
  Anahtar kelime say.: 20
  KW yogunlugu       : 0.0778
  Capraz kopya say.  : 0
  KW parmak izi esm. : 0


[3. TEST METNİ]: Kaufen Sie billige Schuhe online. Billige Schuhe s...

  Organiklik Skoru : 0.9807  [#############################-]
  Karar            : Organik

Cikarilan Ozellikler:
  Metin uzunlugu     : 229
  Anahtar kelime say.: 16
  KW yogunlugu       : 0.0696
  Capraz kopya say.  : 0
  KW parmak izi esm. : 0


[4. TEST METNİ]: Canlı bahis casino 